# 02 Experiments

Короткий ноутбук для честных экспериментов: validation подбирает параметры, test только проверяет итог.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score

from src import config
from src.connector.data_fetcher import load_all_price_data
from src.features.event_detector import detect_events
from src.features.feature_pipeline import generate_features
from src.models.sequence_models import load_model, load_model_with_config
from src.models.training import train_direction_model
from src.strategy.backtest import build_trades, calculate_trade_metrics
from src.strategy.signal_generator import add_labels_for_metrics, generate_rule_based_signal_history, generate_signal_history

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

## Настройки

F1 здесь не главный критерий: он может выглядеть хорошо, когда модель почти всегда выбирает один класс. Для выбора эпохи лучше начать с `balanced_accuracy`.

In [ ]:
# Основные параметры эксперимента.
RUN_TRAINING = False
MODEL_TYPES = ["gru", "lstm"]
DATASET_MODE = "event"

SELECTION_METRIC = "balanced_accuracy"  # accuracy / balanced_accuracy / f1
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 128
DROPOUT = 0.25
NUM_LAYERS = 2

Q_CANDLES = 2000
HORIZON = config.DEFAULT_HORIZON_CANDLES
LABEL_THRESHOLD = config.DEFAULT_LABEL_THRESHOLD

CONFIDENCE_GRID = np.round(np.arange(0.50, 0.71, 0.01), 2)
TP_GRID = [0.0005, 0.0008, 0.0010, 0.0012]
SL_GRID = [0.0005, 0.0008, 0.0010, 0.0012]
MIN_TRADES = 30

VALID_START = pd.Timestamp(config.TRAIN_END_DATE)
TEST_START = pd.Timestamp(config.VALID_END_DATE)

In [ ]:
# Загружаем данные один раз.
price_df, loaded_files = load_all_price_data(config.DATA_DIR)
prepared_df = detect_events(generate_features(price_df))

print(f"CSV файлов: {len(loaded_files)}")
print(f"Свечей: {len(price_df):,}")
print(f"Подготовленных строк: {len(prepared_df):,}")
print(f"Event=1: {int(prepared_df['event'].sum()):,}")
print(f"Период: {prepared_df.index.min()} -> {prepared_df.index.max()}")

## Обучение

По умолчанию выключено. Включи `RUN_TRAINING = True`, если хочешь переобучить модели с выбором лучшей эпохи по `balanced_accuracy`.

In [ ]:
def save_model_config(model_type: str, event_only: bool):
    paths = config.EVENT_CONFIG_PATHS if event_only else config.FULL_CONFIG_PATHS
    config_path = paths[model_type]
    config_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {
            "model_type": model_type,
            "hidden_size": HIDDEN_SIZE,
            "dropout": DROPOUT,
            "num_layers": NUM_LAYERS,
            "selection_metric": SELECTION_METRIC,
        },
        config_path,
    )
    return config_path


if RUN_TRAINING:
    train_rows = []
    for model_type in MODEL_TYPES:
        model_path = config.EVENT_MODEL_PATHS[model_type]
        scaler_path = config.EVENT_SCALER_PATHS[model_type]

        result = train_direction_model(
            price_df,
            model_type=model_type,
            event_only=True,
            label_threshold=LABEL_THRESHOLD,
            horizon=HORIZON,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            model_path=model_path,
            scaler_path=scaler_path,
            hidden_size=HIDDEN_SIZE,
            dropout=DROPOUT,
            num_layers=NUM_LAYERS,
            selection_metric=SELECTION_METRIC,
        )
        config_path = save_model_config(model_type, event_only=True)

        train_rows.append(
            {
                "model": model_type,
                "best_valid_metric": SELECTION_METRIC,
                "best_valid_score": result["best_valid_score"],
                "test_accuracy": result["test_metrics"]["accuracy"],
                "test_balanced_accuracy": result["test_metrics"]["balanced_accuracy"],
                "test_f1": result["test_metrics"]["f1"],
                "train_samples": result["train_samples"],
                "valid_samples": result["valid_samples"],
                "test_samples": result["test_samples"],
                "model_path": str(model_path),
                "config_path": str(config_path),
            }
        )

    display(pd.DataFrame(train_rows))
else:
    print("Обучение пропущено. Используются уже сохраненные модели.")

## Функции оценки

Сначала делаем сырые прогнозы при confidence 0.50, потом применяем пороги отдельно. Так быстрее гонять сетку.

In [ ]:
def localize_like_index(ts: pd.Timestamp, index: pd.Index) -> pd.Timestamp:
    if getattr(index, "tz", None) is not None and ts.tzinfo is None:
        return ts.tz_localize(index.tz)
    return ts


VALID_START_TS = localize_like_index(VALID_START, prepared_df.index)
TEST_START_TS = localize_like_index(TEST_START, prepared_df.index)


def load_event_model_and_scaler(model_type: str):
    model_path = config.EVENT_MODEL_PATHS[model_type]
    scaler_path = config.EVENT_SCALER_PATHS[model_type]
    config_path = config.EVENT_CONFIG_PATHS[model_type]

    if config_path.exists():
        model = load_model_with_config(model_path, config_path)
    else:
        model = load_model(model_path, model_type=model_type)
    scaler = joblib.load(scaler_path)
    return model, scaler


def apply_confidence_threshold(signals: pd.DataFrame, threshold: float, require_event: bool = True) -> pd.DataFrame:
    if signals.empty:
        return signals.copy()

    result = signals.copy()
    event_allowed = result["event"].eq(1) if require_event else pd.Series(True, index=result.index)
    confident = result["confidence"].ge(threshold)

    result["decision"] = "NO TRADE"
    result.loc[event_allowed & confident & result["prediction"].eq("UP"), "decision"] = "BUY"
    result.loc[event_allowed & confident & result["prediction"].eq("DOWN"), "decision"] = "SELL"
    return result


def trim_horizon_boundary(signals: pd.DataFrame, period_end: pd.Timestamp | None, horizon: int) -> pd.DataFrame:
    if signals.empty or period_end is None:
        return signals.copy()

    positions = prepared_df.index.get_indexer(signals["time"], method="nearest")
    exit_positions = np.minimum(positions + horizon, len(prepared_df) - 1)
    keep = prepared_df.index[exit_positions] < period_end
    return signals.loc[keep].copy()


def select_period(signals: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp | None = None, q_candles: int | None = None) -> pd.DataFrame:
    result = signals[signals["time"].ge(start)].copy()
    if end is not None:
        result = result[result["time"].lt(end)].copy()
    result = trim_horizon_boundary(result, end, HORIZON)
    if q_candles is not None:
        result = result.tail(q_candles).copy()
    return result


def classification_metrics(signals: pd.DataFrame) -> dict:
    labeled = add_labels_for_metrics(signals, prepared_df, HORIZON, threshold=LABEL_THRESHOLD)
    labeled = labeled[labeled["event"].eq(1)].copy()

    if labeled.empty:
        return {"accuracy": 0.0, "balanced_accuracy": 0.0, "f1": 0.0, "precision": 0.0, "recall": 0.0, "classified": 0}

    y_true = labeled["actual"]
    y_pred = labeled["prediction"]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, pos_label="UP", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label="UP", zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label="UP", zero_division=0),
        "classified": len(labeled),
    }


def evaluate_strategy(name: str, raw_signals: pd.DataFrame, confidence: float, tp: float, sl: float) -> dict:
    signals = apply_confidence_threshold(raw_signals, confidence, require_event=True)
    trades = build_trades(signals, prepared_df, horizon=HORIZON, tp_threshold=tp, sl_threshold=sl)
    trade_metrics = calculate_trade_metrics(trades)
    clf = classification_metrics(signals)

    return {
        "strategy": name,
        "confidence": confidence,
        "tp": tp,
        "sl": sl,
        "accuracy": clf["accuracy"],
        "balanced_accuracy": clf["balanced_accuracy"],
        "f1": clf["f1"],
        "precision": clf["precision"],
        "recall": clf["recall"],
        "classified": clf["classified"],
        "trades": trade_metrics["Trades"],
        "total_return": trade_metrics["Total Return"],
        "winrate": trade_metrics["Win Rate"],
        "profit_factor": trade_metrics["Profit Factor"],
        "max_drawdown": trade_metrics["Max Drawdown"],
        "avg_trade": trade_metrics["Average Trade"],
    }


def selection_score(row: pd.Series) -> float:
    # Штрафуем слишком редкие сделки и угадывание на уровне монетки.
    score = row["total_return"]
    if row["trades"] < MIN_TRADES:
        score -= 10.0
    if row["balanced_accuracy"] < 0.5:
        score -= 1.0
    return score

In [ ]:
# Сырые сигналы для validation и test.
raw_by_model = {}

for model_type in MODEL_TYPES:
    model, scaler = load_event_model_and_scaler(model_type)
    raw = generate_signal_history(
        price_df,
        threshold=0.50,
        max_rows=len(prepared_df),
        model_type=model_type,
        require_event=True,
        model=model,
        scaler=scaler,
    )
    raw_by_model[model_type] = {
        "valid": select_period(raw, VALID_START_TS, TEST_START_TS),
        "test": select_period(raw, TEST_START_TS, None, Q_CANDLES),
    }
    print(model_type, "valid", len(raw_by_model[model_type]["valid"]), "test", len(raw_by_model[model_type]["test"]))

rule_raw = generate_rule_based_signal_history(price_df, max_rows=len(prepared_df))
rule_periods = {
    "valid": select_period(rule_raw, VALID_START_TS, TEST_START_TS),
    "test": select_period(rule_raw, TEST_START_TS, None, Q_CANDLES),
}
print("rule", "valid", len(rule_periods["valid"]), "test", len(rule_periods["test"]))

## Подбор на validation

Тут ищем не максимальный F1, а торговую комбинацию: доходность, достаточное число сделок и balanced accuracy выше 0.5.

In [ ]:
valid_rows = []

for model_type, periods in raw_by_model.items():
    for confidence in CONFIDENCE_GRID:
        for tp in TP_GRID:
            for sl in SL_GRID:
                valid_rows.append(evaluate_strategy(f"Event + {model_type.upper()}", periods["valid"], confidence, tp, sl))

valid_df = pd.DataFrame(valid_rows)
valid_df["selection_score"] = valid_df.apply(selection_score, axis=1)
valid_top = valid_df.sort_values(
    ["selection_score", "profit_factor", "balanced_accuracy", "trades"],
    ascending=[False, False, False, False],
).head(20)

display(valid_top)

## Финальная проверка на test

Берем лучшие параметры с validation и проверяем на последних `Q_CANDLES` свечах test.

In [ ]:
best_by_strategy = (
    valid_df.sort_values(["selection_score", "profit_factor", "balanced_accuracy"], ascending=[False, False, False])
    .groupby("strategy", as_index=False)
    .head(1)
)

test_rows = []
for _, row in best_by_strategy.iterrows():
    model_type = row["strategy"].split("+")[-1].strip().lower()
    test_rows.append(
        evaluate_strategy(
            row["strategy"],
            raw_by_model[model_type]["test"],
            float(row["confidence"]),
            float(row["tp"]),
            float(row["sl"]),
        )
    )

# Бейзлайн не подбираем по confidence: это правило, а не ML-модель.
test_rows.append(evaluate_strategy("Rule baseline", rule_periods["test"], 0.50, config.DEFAULT_TP_THRESHOLD, config.DEFAULT_SL_THRESHOLD))

test_df = pd.DataFrame(test_rows).sort_values(["total_return", "profit_factor"], ascending=[False, False])
display(test_df)

## Быстрая диагностика confidence

Если при 0.55 или 0.60 сделок почти нет, значит confidence плохо калиброван. Тогда лучше не поднимать порог вслепую, а менять обучение/лейблы.

In [ ]:
diagnostic_rows = []

for model_type, periods in raw_by_model.items():
    for confidence in [0.50, 0.52, 0.55, 0.57, 0.60]:
        diagnostic_rows.append(
            evaluate_strategy(
                f"Event + {model_type.upper()}",
                periods["test"],
                confidence,
                config.DEFAULT_TP_THRESHOLD,
                config.DEFAULT_SL_THRESHOLD,
            )
        )

diagnostic_df = pd.DataFrame(diagnostic_rows)
display(diagnostic_df[["strategy", "confidence", "accuracy", "balanced_accuracy", "f1", "classified", "trades", "total_return", "winrate", "profit_factor"]])